# 黑白图像形态学实验：膨胀、腐蚀、开运算与闭运算（学生练习版）

本 Notebook 是学生练习版：背景知识、三类二值图像生成、可视化和对比实验已经保留；核心形态学算法函数被改为 `TODO`，需要学生补全。

本实验面向二值图像，演示数字图像处理中常用的形态学操作：

1. **膨胀 Dilation**
2. **腐蚀 Erosion**
3. **开运算 Opening**
4. **闭运算 Closing**

实验将自动生成三类二值图像：

- 图像一：简单几何图形。
- 图像二：带噪声的二值图像。
- 图像三：存在孔洞或断裂的二值图像。

随后使用不同大小的结构元素进行实验，观察结构元素大小如何影响形态学处理结果。

## 1. 相关背景知识

### 1.1 二值图像

二值图像只有两类像素：

- `1` 或白色：前景目标。
- `0` 或黑色：背景。

形态学操作通常直接作用于前景目标的形状。它不关心灰度变化，而是关心目标区域如何扩张、收缩、断开或连接。

### 1.2 结构元素

结构元素可以理解为一个小模板，例如 `3 × 3` 方形、十字形或圆盘形。形态学运算会让结构元素在图像上滑动，并根据结构元素覆盖区域的像素情况决定输出结果。

本实验统一采用 **方形结构元素**，并比较不同大小的处理效果。方形结构元素在水平、垂直和对角方向都会参与运算，适合观察结构元素尺寸变化对图像形态的影响。

### 1.3 膨胀

膨胀会让前景目标变大。结构元素覆盖范围内只要有一个前景像素，输出中心位置就变成前景。

常见效果：

- 填补小缝隙。
- 连接距离较近的断裂区域。
- 增大目标边界。

### 1.4 腐蚀

腐蚀会让前景目标变小。结构元素覆盖范围内必须全部满足前景条件，输出中心位置才保持为前景。

常见效果：

- 去掉细小突出物。
- 消除孤立噪声点。
- 使目标边界向内收缩。

### 1.5 开运算

开运算定义为：

`开运算 = 先腐蚀，再膨胀`

常见效果：

- 去除小的白色噪声点。
- 断开细小连接。
- 保持主要目标的大致形状。

### 1.6 闭运算

闭运算定义为：

`闭运算 = 先膨胀，再腐蚀`

常见效果：

- 填补小孔洞。
- 连接窄小断裂。
- 平滑目标轮廓。

## 2. 实验对象设计

本实验不依赖外部图片文件，而是直接用程序生成三类二值图像。

### 图像一：简单几何图形

包含矩形、圆形、三角形和线段，用于观察基本形状在膨胀、腐蚀后的变化。

### 图像二：带噪声的二值图像

在基础目标上加入白色噪声点和黑色空洞噪声，用于观察开运算和闭运算对噪声的处理效果。

### 图像三：存在孔洞或断裂的二值图像

包含环形孔洞、断裂线段和有缺口的目标，用于观察闭运算对孔洞和断裂的修复能力。

实验将统一采用方形结构元素，并比较不同结构元素大小：

- `3 × 3`
- `5 × 5`
- `9 × 9`

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def show_binary_image(image, title="", ax=None):
    """显示单张二值图像。

    参数：
    - image：二维 NumPy 数组，0 表示背景，1 表示前景。
    - title：图像标题。
    - ax：Matplotlib 坐标轴对象；如果为 None，则创建新图。
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")


def show_image_grid(images, titles, main_title="", cols=5, figsize=(14, 8)):
    """以网格方式显示多张二值图像。"""
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, image, title in zip(axes, images, titles):
        show_binary_image(image, title, ax)
    for ax in axes[len(images):]:
        ax.axis("off")
    fig.suptitle(main_title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def create_simple_geometric_image(size=128):
    """生成图像一：简单几何图形。"""
    image = np.zeros((size, size), dtype=np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]

    # 矩形
    image[18:48, 16:52] = 1

    # 圆形
    circle = (xx - 92) ** 2 + (yy - 34) ** 2 <= 18 ** 2
    image[circle] = 1

    # 三角形，使用简单不等式构造
    triangle = (yy >= 72) & (yy <= 112) & (xx >= 18 + (yy - 72) * 0.45) & (xx <= 58 - (yy - 72) * 0.45)
    image[triangle] = 1

    # 线段
    image[84:88, 76:116] = 1
    image[88:108, 96:100] = 1
    return image


def create_noisy_binary_image(size=128, noise_ratio=0.035):
    """生成图像二：带噪声的二值图像。

    参数：
    - size：图像边长。
    - noise_ratio：噪声比例，值越大噪声越多。
    """
    image = np.zeros((size, size), dtype=np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]

    body = ((xx - 64) / 36) ** 2 + ((yy - 66) / 26) ** 2 <= 1
    image[body] = 1
    image[35:92, 28:52] = 1
    image[45:98, 78:102] = 1

    # 白色椒盐噪声：背景中随机出现小白点
    white_noise = np.random.random((size, size)) < noise_ratio
    image[white_noise] = 1

    # 黑色噪声：目标内部随机出现小黑洞
    black_noise = (np.random.random((size, size)) < noise_ratio) & (image == 1)
    image[black_noise] = 0
    return image


def create_hole_broken_image(size=128):
    """生成图像三：存在孔洞或断裂的二值图像。"""
    image = np.zeros((size, size), dtype=np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]

    # 环形目标：中间有孔洞
    outer = (xx - 38) ** 2 + (yy - 42) ** 2 <= 24 ** 2
    inner = (xx - 38) ** 2 + (yy - 42) ** 2 <= 10 ** 2
    image[outer & ~inner] = 1

    # 有缺口的矩形
    image[28:70, 78:112] = 1
    image[42:56, 90:100] = 0
    image[28:70, 94:98] = 0

    # 断裂线段
    image[92:98, 18:52] = 1
    image[92:98, 60:96] = 1
    image[100:106, 40:76] = 1
    image[100:106, 84:112] = 1
    return image


image_simple = create_simple_geometric_image()
image_noisy = create_noisy_binary_image()
image_hole_broken = create_hole_broken_image()

show_image_grid(
    [image_simple, image_noisy, image_hole_broken],
    ["图像一：简单几何图形", "图像二：带噪声二值图像", "图像三：孔洞或断裂图像"],
    main_title="三类实验二值图像",
    cols=3,
    figsize=(12, 4),
)

## 3. 形态学运算程序

下面从零实现二值图像的膨胀、腐蚀、开运算和闭运算。为了便于教学，本实验不调用 OpenCV 或 scikit-image，而是直接使用 `numpy` 编写核心逻辑。

约定：

- 图像中 `1` 表示白色前景。
- 图像中 `0` 表示黑色背景。
- 结构元素中 `True` 表示参与运算的位置。

### 3.1 算法补全步骤

请按下面顺序完成黑白二值形态学算法：

1. 补全 `create_structuring_element(size)`，生成指定大小的方形结构元素。
2. 补全 `binary_dilation(image, se)`，实现二值膨胀：邻域中只要有前景，中心输出前景。
3. 补全 `binary_erosion(image, se)`，实现二值腐蚀：结构元素覆盖位置必须全部为前景，中心才输出前景。
4. 补全 `binary_opening(image, se)`，按“先腐蚀，再膨胀”实现开运算。
5. 补全 `binary_closing(image, se)`，按“先膨胀，再腐蚀”实现闭运算。
6. 运行后续对比单元，观察不同结构元素大小对结果的影响。

In [ ]:
def create_structuring_element(size=3):
    """创建方形结构元素。

    参数：
    - size：结构元素大小，建议使用奇数，例如 3、5、9。

    返回：
    - 布尔数组，True 表示结构元素参与运算的位置。
    """
    # TODO 1：判断 size 是否为奇数，如果不是奇数则抛出 ValueError。
    # TODO 2：返回一个形状为 (size, size)、元素全为 True 的布尔数组。
    raise NotImplementedError("请补全 create_structuring_element 函数")


def binary_dilation(image, se):
    """二值膨胀。

    参数：
    - image：输入二值图像。
    - se：结构元素。

    返回：
    - 膨胀后的二值图像。
    """
    # TODO 1：根据结构元素大小对 image 做 0 填充。
    # TODO 2：遍历图像中每个像素，取出对应邻域 region。
    # TODO 3：如果 region 中 se 为 True 的位置存在前景 1，则输出 1，否则输出 0。
    # TODO 4：返回膨胀结果。
    raise NotImplementedError("请补全 binary_dilation 函数")


def binary_erosion(image, se):
    """二值腐蚀。

    参数：
    - image：输入二值图像。
    - se：结构元素。

    返回：
    - 腐蚀后的二值图像。
    """
    # TODO 1：根据结构元素大小对 image 做 0 填充。
    # TODO 2：遍历图像中每个像素，取出对应邻域 region。
    # TODO 3：如果 region 中 se 为 True 的位置全部为前景 1，则输出 1，否则输出 0。
    # TODO 4：返回腐蚀结果。
    raise NotImplementedError("请补全 binary_erosion 函数")


def binary_opening(image, se):
    """二值开运算：先腐蚀，再膨胀。"""
    # TODO：调用 binary_erosion 和 binary_dilation 实现开运算。
    raise NotImplementedError("请补全 binary_opening 函数")


def binary_closing(image, se):
    """二值闭运算：先膨胀，再腐蚀。"""
    # TODO：调用 binary_dilation 和 binary_erosion 实现闭运算。
    raise NotImplementedError("请补全 binary_closing 函数")

## 4. 基本形态学操作对比

下面对三类图像分别执行：

- 原图
- 膨胀
- 腐蚀
- 开运算
- 闭运算

这里先统一使用 `5 × 5` 方形结构元素，观察四种运算的基本效果。

In [ ]:
def compare_basic_operations(image, image_name, se_shape="square", se_size=5):
    """对单张图像展示四种基本形态学运算。"""
    se = create_structuring_element(se_size)
    results = [
        image,
        binary_dilation(image, se),
        binary_erosion(image, se),
        binary_opening(image, se),
        binary_closing(image, se),
    ]
    titles = [
        "原图",
        "膨胀",
        "腐蚀",
        "开运算",
        "闭运算",
    ]
    show_image_grid(
        results,
        titles,
        main_title=f"{image_name}：方形结构元素，大小 {se_size}×{se_size}",
        cols=5,
        figsize=(15, 3.4),
    )


compare_basic_operations(image_simple, "图像一：简单几何图形")
compare_basic_operations(image_noisy, "图像二：带噪声二值图像")
compare_basic_operations(image_hole_broken, "图像三：孔洞或断裂图像")

## 5. 不同结构元素大小的影响

结构元素越大，对图像形状的改变越明显：

- 膨胀会让目标扩张得更多。
- 腐蚀会让目标收缩得更多。
- 开运算更容易去掉较大的噪声或细小连接。
- 闭运算更容易填补较大的孔洞或断裂。

下面固定结构元素为方形，比较 `3 × 3`、`5 × 5`、`9 × 9` 的处理结果。

In [ ]:
def compare_sizes(image, image_name, operation_name, operation_func, sizes=(3, 5, 9)):
    """比较不同结构元素大小对同一运算的影响。"""
    results = [image]
    titles = ["原图"]
    for size in sizes:
        se = create_structuring_element(size)
        results.append(operation_func(image, se))
        titles.append(f"方形 {size}×{size}")
    show_image_grid(
        results,
        titles,
        main_title=f"{image_name}：{operation_name}中不同结构元素大小的影响",
        cols=len(results),
        figsize=(14, 3.6),
    )


compare_sizes(image_noisy, "带噪声二值图像", "开运算", binary_opening)
compare_sizes(image_hole_broken, "孔洞或断裂图像", "闭运算", binary_closing)
compare_sizes(image_simple, "简单几何图形", "腐蚀", binary_erosion)

## 6. 实验思考

完成实验后，可以思考下面的问题：

1. 膨胀和腐蚀分别会让前景目标发生什么变化？
2. 为什么开运算适合去除小的白色噪声点？
3. 为什么闭运算适合填补小孔洞或连接小断裂？
4. 当结构元素从 `3 × 3` 增大到 `9 × 9` 时，图像变化为什么更明显？
5. 如果目标中有细长线段，使用较大的结构元素进行腐蚀会发生什么？